# Composite Return-to-Dispersion Ranking (Sharpe-style) of Five Tech Companies

**Coding for Finance**

An investor wants to rank five publicly traded technology companies — **NVIDIA, Meta, Netflix, Adobe, Intel** — combining expected performance and stability into a single score, using FY2021–FY2025 company data (source: stockanalysis.com).

**Criteria:** Revenue Growth, Operating Margin, Return on Assets, EPS (all *benefit* criteria) and Debt-to-Equity (the one *cost* criterion).

**What the score is, and what it is not.** It mimics the structure of a Sharpe ratio (performance divided by dispersion), but it is *not* a Sharpe ratio: numerator and denominator come from different fundamentals, not from one return series. Performance is the weighted average of the five normalized criteria; dispersion is the weighted year-to-year variability of the criteria for which a complete yearly series exists for every company.

**Data limitations, stated explicitly:**
- *Revenue growth* has 4 yearly observations (FY2022–FY2025), the other criteria have 5.
- *ROA*: complete series for NVIDIA and Intel; for Meta, Netflix and Adobe only the latest value was available, so it is used as a level (average) only.
- *Debt-to-Equity*: complete series for NVIDIA, Meta and Netflix; Adobe uses a single approximate value (60.0) repeated across years; Intel was not found and is imputed with the peer average. Both are used as levels only, and the robustness section checks the ranking without this criterion.
- Because a repeated value has no real variability, *dispersion is measured only on Revenue Growth, Operating Margin and EPS*, for which every company has a genuine yearly series. Otherwise the companies with repeated values would look artificially stable.
- EPS in dollars depends on each company's number of shares, so it is not strictly comparable across companies.

**Note on the data and results.** The figures come from stockanalysis.com and are used for illustration only. Several values are incomplete or approximated (see the limitations above), so the ranking demonstrates the method and is neither a valuation of the companies nor investment advice.

## 1. Data

Five criteria, five companies, FY2021–FY2025.

In [1]:
import numpy as np
import pandas as pd

companies = np.array(["NVIDIA", "Meta", "Netflix", "Adobe", "Intel"])

revenue = np.array([
    [16675,  26914,  26974,  60922,  130497],
    [117929, 116609, 134902, 164501, 200966],
    [29698,  31616,  33723,  39001,  45183],
    [15785,  17606,  19409,  21505,  23769],
    [79024,  63054,  54228,  53101,  52853],
])
revenue_growth = (revenue[:, 1:] / revenue[:, :-1] - 1) * 100

operating_margin = np.array([
    [27.18, 37.31, 20.68, 54.12, 62.42],
    [39.65, 24.82, 34.66, 42.18, 41.44],
    [20.86, 17.82, 20.62, 26.71, 29.49],
    [36.76, 34.64, 34.26, 31.35, 36.63],
    [24.62, 3.70,  0.17,  -21.99, -4.19],
])

roa = np.array([
    [18.79, 30.86, 19.6, 60.45, 83.76],
    [20.90, 20.90, 20.90, 20.90, 20.90],   # Meta: current value only, broadcast
    [17.90, 17.90, 17.90, 17.90, 17.90],   # Netflix: current value only, broadcast
    [24.17, 24.17, 24.17, 24.17, 24.17],   # Adobe: current value only, broadcast
    [14.43, 13.91, 4.33,  2.37,  -0.29],
])

eps = np.array([
    [0.18, 0.39, 0.18, 1.21, 2.97],
    [13.77, 8.59, 14.87, 23.86, 23.49],
    [1.12, 1.00, 1.20, 1.98, 2.53],
    [10.02, 10.10, 11.82, 12.36, 16.70],
    [4.86, 1.94, 0.40, -4.38, -0.06],
])

debt_equity = np.array([
    [39.06, 43.92, 47.99, 22.28, 12.58],
    [10.21, 20.06, 23.25, 25.80, 37.60],
    [92.71, 69.08, 68.69, 55.77, 50.59],
    [60.0,  60.0,  60.0,  60.0,  60.0],         # Adobe: single approximate value, repeated
    [np.nan, np.nan, np.nan, np.nan, np.nan],   # Intel: not found -> imputed below
])
peer_avg_de = np.nanmean(debt_equity[:4])
debt_equity = np.where(np.isnan(debt_equity), peer_avg_de, debt_equity)  # Intel = peer average

## 2. Mean and variance per criterion (2021–2025)

In [2]:
criteria_names = ["Revenue Growth (%)", "Operating Margin (%)", "ROA (%)",
                  "EPS ($)", "Debt-to-Equity (%)"]
mean_matrix = np.column_stack([revenue_growth.mean(axis=1), operating_margin.mean(axis=1),
                                roa.mean(axis=1), eps.mean(axis=1), debt_equity.mean(axis=1)])
var_matrix = np.column_stack([revenue_growth.var(axis=1), operating_margin.var(axis=1),
                               roa.var(axis=1), eps.var(axis=1), debt_equity.var(axis=1)])
is_benefit = np.array([True, True, True, True, False])

print(pd.DataFrame(mean_matrix, index=companies, columns=criteria_names).round(2))

         Revenue Growth (%)  Operating Margin (%)  ROA (%)  EPS ($)  \
NVIDIA                75.42                 40.34    42.69     0.99   
Meta                  14.67                 36.55    20.90    16.92   
Netflix               11.16                 23.10    17.90     1.57   
Adobe                 10.78                 34.73    24.17    12.20   
Intel                 -9.19                  0.46     6.95     0.55   

         Debt-to-Equity (%)  
NVIDIA                33.17  
Meta                  23.38  
Netflix               67.37  
Adobe                 60.00  
Intel                 45.98  


## 3. Normalization

Because criteria are on different scales, each criterion is min-max normalized onto [0, 1] before combining — benefit criteria as $(x-\min)/(\max-\min)$, the cost criterion (Debt-to-Equity) with the direction flipped. The variances are put on the same scale (min-max is a linear map, so each variance is divided by the squared range of its criterion); otherwise the numerator and the denominator of the score would be expressed in different units.

In [3]:
# Min-Max normalization (vectorized, benefit vs. cost criteria in one line)
col_max, col_min = mean_matrix.max(axis=0), mean_matrix.min(axis=0)
col_range = col_max - col_min
norm_mean = np.where(is_benefit, (mean_matrix - col_min) / col_range,
                      (col_max - mean_matrix) / col_range)

# Variances on the same scale as the normalized means (squared range)
norm_var = var_matrix / col_range ** 2

print(pd.DataFrame(norm_mean, index=companies, columns=criteria_names).round(4))

         Revenue Growth (%)  Operating Margin (%)  ROA (%)  EPS ($)  \
NVIDIA               1.0000                1.0000   1.0000   0.0265   
Meta                 0.2820                0.9049   0.3903   1.0000   
Netflix              0.2404                0.5677   0.3064   0.0620   
Adobe                0.2360                0.8592   0.4818   0.7118   
Intel                0.0000                0.0000   0.0000   0.0000   

         Debt-to-Equity (%)  
NVIDIA               0.7776  
Meta                 1.0000  
Netflix              0.0000  
Adobe                0.1675  
Intel                0.4863  


## 4. Composite Sharpe-style ranking

$$\text{Sharpe}_i = \frac{\sum_j w_j\, r_{ij}}{\sqrt{\sum_{j\in D} \tilde w_j^{\,2}\, \sigma_{ij}^2}}, \qquad w = (0.20, 0.25, 0.25, 0.20, 0.10)$$

where $r_{ij}$ and $\sigma^2_{ij}$ are the normalized mean and variance of criterion $j$ for company $i$, $D$ = {Revenue Growth, Operating Margin, EPS} (the criteria with a genuine yearly series for every company) and $\tilde w_j$ are the weights rescaled to sum to one on $D$. The weights favor Operating Margin and ROA slightly (profitability), and de-emphasize Debt-to-Equity. No risk-free rate is subtracted: the numerator is a normalized score, not a return.

In [4]:
# Composite "Sharpe Ratio": weighted normalized return over weighted risk.
weights = np.array([0.20, 0.25, 0.25, 0.20, 0.10])

# Risk is measured only on the criteria with a genuine yearly series for every company:
# Revenue Growth, Operating Margin, EPS (ROA and Debt-to-Equity enter as levels only)
dispersion_criteria = [0, 1, 3]
weights_disp = weights[dispersion_criteria] / weights[dispersion_criteria].sum()

weighted_return = norm_mean @ weights
weighted_risk = np.sqrt((weights_disp ** 2) @ norm_var[:, dispersion_criteria].T)
sharpe = weighted_return / weighted_risk

df_sharpe = pd.DataFrame({"Weighted Return": weighted_return, "Weighted Risk": weighted_risk,
                           "Sharpe": sharpe}, index=companies).sort_values("Sharpe", ascending=False)
print(df_sharpe.round(4))

         Weighted Return  Weighted Risk   Sharpe
Adobe             0.5416         0.0496  10.9290
Netflix           0.2790         0.0462   6.0425
Meta              0.6802         0.1319   5.1579
NVIDIA            0.7831         0.2373   3.2999
Intel             0.0486         0.1579   0.3080


## 5. Robustness

Does the ranking depend on the weights, or on the imputed Debt-to-Equity values?

In [5]:
variants = {"Base weights": weights,
            "Equal weights": np.ones(5),
            "Without Debt-to-Equity": np.array([0.20, 0.25, 0.25, 0.20, 0.0])}

ranks = {}
for name, w in variants.items():
    w = w / w.sum()
    w_d = w[dispersion_criteria] / w[dispersion_criteria].sum()
    s = (norm_mean @ w) / np.sqrt((w_d ** 2) @ norm_var[:, dispersion_criteria].T)
    ranks[name] = pd.Series(s, index=companies).rank(ascending=False).astype(int)

print(pd.DataFrame(ranks).sort_values("Base weights"))

         Base weights  Equal weights  Without Debt-to-Equity
Adobe               1              1                       1
Netflix             2              2                       2
Meta                3              3                       3
NVIDIA              4              4                       4
Intel               5              5                       5


## 6. Interpretation and limitations

- **Adobe and Netflix lead** because their revenue growth, operating margin and EPS move little from year to year: the score rewards stability, and their risk is the lowest of the group.
- **NVIDIA has the highest weighted return but ranks fourth.** Its growth and margins swing widely (revenue growth ranges from almost zero to over 100% in a year), and the risk term penalizes upward swings as much as downward ones. It is a measure of variability, not of downside risk.
- **Intel ranks last**, with the lowest weighted return (operating margin and ROA turned negative in the latest years) and above-average variability.
- **The order is unchanged** with equal weights and without the Debt-to-Equity criterion, so the conclusion does not depend on the weight choice or on the imputed values.
- **Limitation.** Unlike a textbook Sharpe ratio (numerator and denominator from the *same* return series), here "return" and "risk" come from different criteria forced onto a common scale by min-max normalization. With only five companies the scale is sensitive to the extremes (NVIDIA's growth sets the range for everyone), dispersion is computed on 4–5 yearly observations, and ROA and Debt-to-Equity enter as levels only because their yearly histories were incomplete. A more rigorous version would use complete yearly series for all criteria.